# Narrow `account users` access to `workshop_guy_catalog.fe_bar_a0` (Step 8 follow-up)

User-approved plan (conversation 2326365276521178, 2026-11-04). `account users` held ALL PRIVILEGES, MANAGE and EXTERNAL USE SCHEMA on the shared catalog `workshop_guy_catalog`. Through inheritance, that exposed `scored_transactions`, `txn_features_gold`, the bronze tables and the `staging` volume, and let any user drop filters and masks. A schema cannot block inherited catalog grants, so:

1. Grant ALL PRIVILEGES, MANAGE and EXTERNAL USE SCHEMA directly on every other schema (same effective access for other teams and their service principals).
2. On `fe_bar_a0`, grant only USE SCHEMA plus SELECT on governed or safe objects.
3. Replace the catalog grant with USE CATALOG, CREATE SCHEMA and BROWSE.

The owner identity (alex.owen, which runs the pipeline, job and Lakebase sync) keeps its own ALL PRIVILEGES and MANAGE. Nothing else is changed.

In [0]:
import datetime
CAT, OWN = "workshop_guy_catalog", "fe_bar_a0"
AU = "account users"
# fe_bar_a0 objects account users may SELECT afterwards (governed or safe); everything else in fe_bar_a0 is closed
SAFE = ["fact_transactions_silver", "dim_customer_silver", "dim_merchant_silver", "customer_risk_profile_gold", "risk_score_calibration"]
print("run_at_utc:", datetime.datetime.now(datetime.timezone.utc).isoformat(timespec="seconds"), "| user:", spark.sql("SELECT current_user()").first()[0])

OTHERS = [r.schema_name for r in spark.sql(f"SELECT schema_name FROM {CAT}.information_schema.schemata WHERE schema_name NOT IN ('information_schema', '{OWN}') ORDER BY 1").collect()]
print(f"other schemas ({len(OTHERS)}):", OTHERS)

def cat_privs(grantee):
    return sorted(r.privilege_type for r in spark.sql(f"SELECT privilege_type FROM {CAT}.information_schema.catalog_privileges WHERE grantee = '{grantee}'").collect())
def schema_privs(schema, grantee=AU):
    return sorted({(r.privilege_type, r.inherited_from) for r in spark.sql(f"SELECT privilege_type, inherited_from FROM {CAT}.information_schema.schema_privileges WHERE grantee = '{grantee}' AND schema_name = '{schema}'").collect()})
def table_privs(schema, grantee=AU):
    return sorted({(r.table_name, r.privilege_type, r.inherited_from) for r in spark.sql(f"SELECT table_name, privilege_type, inherited_from FROM {CAT}.information_schema.table_privileges WHERE grantee = '{grantee}' AND table_schema = '{schema}'").collect()})

BEFORE = {"catalog": cat_privs(AU), "owner_catalog": cat_privs("alex.owen@databricks.com")}
print("\naccount users on catalog:", BEFORE["catalog"])
print("alex.owen on catalog:   ", BEFORE["owner_catalog"])
for s in OTHERS + [OWN]:
    BEFORE[s] = schema_privs(s)
    print(f"  {s:24} {BEFORE[s]}")
BEFORE["fe_bar_a0_tables"] = table_privs(OWN)
print(f"\nfe_bar_a0 table privileges for account users: {len(BEFORE['fe_bar_a0_tables'])} rows, e.g. {BEFORE['fe_bar_a0_tables'][:3]}")
print("other direct catalog grantees:", [(r.grantee, r.privilege_type) for r in spark.sql(f"SELECT grantee, privilege_type FROM {CAT}.information_schema.catalog_privileges WHERE grantee NOT IN ('{AU}', 'alex.owen@databricks.com')").collect()])

run_at_utc: 2026-11-04T03:35:27+00:00 | user: alex.owen@databricks.com
other schemas (17): ['agent_genie_traces', 'alex_o_test', 'asana_demo', 'genie_agent', 'genie_asana_workshop', 'genie_mlflow_bohao', 'genie_new_traces', 'genie_traces_bo', 'genie_traces_bo_test', 'genie_traces_test', 'genie_workshop', 'memory_pipeline', 'mlflow_traces_bo', 'mlflow_traces_offline', 'rls_demo', 'samraj_test', 'test_permissions']

account users on catalog: ['ALL_PRIVILEGES', 'EXTERNAL_USE_SCHEMA', 'MANAGE']
alex.owen on catalog:    ['ALL_PRIVILEGES', 'MANAGE']
  agent_genie_traces       [('ALL_PRIVILEGES', 'CATALOG'), ('EXTERNAL_USE_SCHEMA', 'CATALOG'), ('MANAGE', 'CATALOG')]
  alex_o_test              [('ALL_PRIVILEGES', 'CATALOG'), ('EXTERNAL_USE_SCHEMA', 'CATALOG'), ('MANAGE', 'CATALOG')]
  asana_demo               [('ALL_PRIVILEGES', 'CATALOG'), ('EXTERNAL_USE_SCHEMA', 'CATALOG'), ('MANAGE', 'CATALOG')]
  genie_agent              [('ALL_PRIVILEGES', 'CATALOG'), ('EXTERNAL_USE_SCHEMA', 'CATALOG'), (

In [0]:
DONE = []
def run(sql):
    spark.sql(sql); DONE.append(sql); print("OK ", sql)

# Step 1: keep other teams' effective access by granting it directly on each of their schemas (before touching the catalog)
for s in OTHERS:
    run(f"GRANT ALL PRIVILEGES ON SCHEMA {CAT}.`{s}` TO `{AU}`")
    run(f"GRANT MANAGE ON SCHEMA {CAT}.`{s}` TO `{AU}`")
    run(f"GRANT EXTERNAL USE SCHEMA ON SCHEMA {CAT}.`{s}` TO `{AU}`")

# Step 2: fe_bar_a0 least privilege for account users
run(f"GRANT USE SCHEMA ON SCHEMA {CAT}.{OWN} TO `{AU}`")
for t in SAFE:
    run(f"GRANT SELECT ON TABLE {CAT}.{OWN}.{t} TO `{AU}`")

# Step 3: replace the catalog grant (grant the narrow set first, then revoke the broad set)
for p in ["USE CATALOG", "CREATE SCHEMA", "BROWSE"]:
    run(f"GRANT {p} ON CATALOG {CAT} TO `{AU}`")
for p in ["ALL PRIVILEGES", "MANAGE", "EXTERNAL USE SCHEMA"]:
    run(f"REVOKE {p} ON CATALOG {CAT} FROM `{AU}`")
# If REVOKE ALL PRIVILEGES also removed the narrow grants, put them back
now = cat_privs(AU)
for p, key in [("USE CATALOG", "USE_CATALOG"), ("CREATE SCHEMA", "CREATE_SCHEMA"), ("BROWSE", "BROWSE")]:
    if key not in now:
        run(f"GRANT {p} ON CATALOG {CAT} TO `{AU}`")
print(f"\n{len(DONE)} statements applied")

OK  GRANT ALL PRIVILEGES ON SCHEMA workshop_guy_catalog.`agent_genie_traces` TO `account users`
OK  GRANT MANAGE ON SCHEMA workshop_guy_catalog.`agent_genie_traces` TO `account users`
OK  GRANT EXTERNAL USE SCHEMA ON SCHEMA workshop_guy_catalog.`agent_genie_traces` TO `account users`
OK  GRANT ALL PRIVILEGES ON SCHEMA workshop_guy_catalog.`alex_o_test` TO `account users`
OK  GRANT MANAGE ON SCHEMA workshop_guy_catalog.`alex_o_test` TO `account users`
OK  GRANT EXTERNAL USE SCHEMA ON SCHEMA workshop_guy_catalog.`alex_o_test` TO `account users`
OK  GRANT ALL PRIVILEGES ON SCHEMA workshop_guy_catalog.`asana_demo` TO `account users`
OK  GRANT MANAGE ON SCHEMA workshop_guy_catalog.`asana_demo` TO `account users`
OK  GRANT EXTERNAL USE SCHEMA ON SCHEMA workshop_guy_catalog.`asana_demo` TO `account users`
OK  GRANT ALL PRIVILEGES ON SCHEMA workshop_guy_catalog.`genie_agent` TO `account users`
OK  GRANT MANAGE ON SCHEMA workshop_guy_catalog.`genie_agent` TO `account users`
OK  GRANT EXTERNAL U

In [0]:
AFTER = {"catalog": cat_privs(AU), "owner_catalog": cat_privs("alex.owen@databricks.com")}
for s in OTHERS + [OWN]:
    AFTER[s] = schema_privs(s)
AFTER["fe_bar_a0_tables"] = table_privs(OWN)
print("account users on catalog:", BEFORE["catalog"], "->", AFTER["catalog"])
print("alex.owen on catalog:    ", BEFORE["owner_catalog"], "->", AFTER["owner_catalog"])
for s in OTHERS + [OWN]:
    print(f"  {s:24} {AFTER[s]}")
print("fe_bar_a0 table privileges for account users:", AFTER["fe_bar_a0_tables"])

keys = lambda privs: {p for p, _ in privs}
checks = [
  ("catalog: account users = USE_CATALOG, CREATE_SCHEMA, BROWSE only", set(AFTER["catalog"]) == {"USE_CATALOG", "CREATE_SCHEMA", "BROWSE"}, AFTER["catalog"]),
  ("catalog: alex.owen still ALL_PRIVILEGES + MANAGE", {"ALL_PRIVILEGES", "MANAGE"} <= set(AFTER["owner_catalog"]), AFTER["owner_catalog"]),
]
for s in OTHERS:
    checks.append((f"{s}: account users keep ALL_PRIVILEGES, MANAGE, EXTERNAL_USE_SCHEMA", {"ALL_PRIVILEGES", "MANAGE", "EXTERNAL_USE_SCHEMA"} <= keys(AFTER[s]), sorted(keys(AFTER[s]))))
checks.append(("fe_bar_a0: account users have USE_SCHEMA only at schema level", keys(AFTER[OWN]) == {"USE_SCHEMA"}, sorted(keys(AFTER[OWN]))))
granted = {t for t, p, _ in AFTER["fe_bar_a0_tables"] if p == "SELECT"}
checks.append(("fe_bar_a0: SELECT only on the 5 governed/safe objects", granted == set(SAFE) and all(p == "SELECT" for _, p, _ in AFTER["fe_bar_a0_tables"]), sorted(granted)))
bronze = [r.table_name for r in spark.sql(f"SELECT table_name FROM {CAT}.information_schema.tables WHERE table_schema = '{OWN}' AND table_name LIKE '%bronze'").collect()]
for t in ["scored_transactions", "txn_features_gold"] + bronze:
    checks.append((f"fe_bar_a0.{t}: no account users privilege", t not in {x for x, _, _ in AFTER["fe_bar_a0_tables"]}, "closed"))
vol = spark.sql(f"SELECT count(*) FROM {CAT}.information_schema.volume_privileges WHERE grantee = '{AU}' AND volume_schema = '{OWN}'").first()[0]
checks.append(("fe_bar_a0.staging volume: no account users privilege", vol == 0, vol))
# The owner identity (pipeline, job, sync run-as) still reads fe_bar_a0
n = spark.table(f"{CAT}.{OWN}.scored_transactions").count()
checks.append(("owner identity still reads scored_transactions", n == 464632, n))
print()
for name, ok, val in checks:
    print(f"{'PASS' if ok else 'FAIL'}  {name}  [{val}]")
print(f"\n{sum(ok for _, ok, _ in checks)}/{len(checks)} checks pass")

account users on catalog: ['ALL_PRIVILEGES', 'EXTERNAL_USE_SCHEMA', 'MANAGE'] -> ['BROWSE', 'CREATE_SCHEMA', 'USE_CATALOG']
alex.owen on catalog:     ['ALL_PRIVILEGES', 'MANAGE'] -> ['ALL_PRIVILEGES', 'MANAGE']
  agent_genie_traces       [('ALL_PRIVILEGES', 'NONE'), ('EXTERNAL_USE_SCHEMA', 'NONE'), ('MANAGE', 'NONE')]
  alex_o_test              [('ALL_PRIVILEGES', 'NONE'), ('EXTERNAL_USE_SCHEMA', 'NONE'), ('MANAGE', 'NONE')]
  asana_demo               [('ALL_PRIVILEGES', 'NONE'), ('EXTERNAL_USE_SCHEMA', 'NONE'), ('MANAGE', 'NONE')]
  genie_agent              [('ALL_PRIVILEGES', 'NONE'), ('EXTERNAL_USE_SCHEMA', 'NONE'), ('MANAGE', 'NONE')]
  genie_asana_workshop     [('ALL_PRIVILEGES', 'NONE'), ('EXTERNAL_USE_SCHEMA', 'NONE'), ('MANAGE', 'NONE')]
  genie_mlflow_bohao       [('ALL_PRIVILEGES', 'NONE'), ('EXTERNAL_USE_SCHEMA', 'NONE'), ('MANAGE', 'NONE')]
  genie_new_traces         [('ALL_PRIVILEGES', 'NONE'), ('EXTERNAL_USE_SCHEMA', 'NONE'), ('MANAGE', 'NONE')]
  genie_traces_bo         